# ML Metadata Lab: Student Performance Dataset

This notebook demonstrates how to use **ML Metadata (MLMD)** concepts to track artifacts, executions, and lineage in an ML pipeline.

## Modifications from Original Lab
- **Dataset**: Uses the *Student Performance in Exams* dataset instead of Chicago Taxi
- **Schema Generation**: Uses pandas-based validation instead of TFDV
- **Metadata Store**: Uses a custom SQLite-based implementation that mirrors the MLMD API, making it compatible with all Python versions and platforms

## Objectives
- Set up a metadata store using SQLite
- Register artifact types, execution types, and context types
- Track inputs, outputs, and executions
- Record events, attributions, and associations
- Query the metadata store to trace lineage

## MLMD Overview

The architecture of the ML Metadata store:

<img src='img/mlmd_overview.png' alt='MLMD Overview' width='600'>

The key components of the MLMD data model are:

- **ArtifactType / Artifact**: Describes the type and instances of data objects (datasets, models, schemas)
- **ExecutionType / Execution**: Describes the type and instances of pipeline steps (training, validation)
- **Event**: Records relationships between artifacts and executions (input/output)
- **ContextType / Context**: Groups related artifacts and executions (experiments, pipeline runs)
- **Attribution**: Links artifacts to contexts
- **Association**: Links executions to contexts

## Imports

In [ ]:
import pandas as pd
import sqlite3
import json
import os
from datetime import datetime

print('All imports successful.')
print('Pandas version:', pd.__version__)

## Custom Metadata Store

Below is a lightweight SQLite-based metadata store that mirrors the core MLMD API. It supports:
- Registering and querying artifact types, execution types, and context types
- Creating and retrieving artifacts, executions, and contexts
- Recording events (input/output relationships)
- Creating attributions and associations
- Lineage queries

In [ ]:
class MetadataStore:
    """A lightweight SQLite-based metadata store that mirrors the ML Metadata API."""
    
    def __init__(self, db_path=':memory:'):
        """Initialize the metadata store with a SQLite backend.
        
        Args:
            db_path: Path to SQLite database file, or ':memory:' for in-memory store.
        """
        self.conn = sqlite3.connect(db_path)
        self.conn.row_factory = sqlite3.Row
        self._create_tables()
    
    def _create_tables(self):
        """Create the database schema for storing metadata."""
        cursor = self.conn.cursor()
        
        # Type tables
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS artifact_types (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                name TEXT UNIQUE NOT NULL,
                properties TEXT DEFAULT '{}'
            )
        ''')
        
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS execution_types (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                name TEXT UNIQUE NOT NULL,
                properties TEXT DEFAULT '{}'
            )
        ''')
        
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS context_types (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                name TEXT UNIQUE NOT NULL,
                properties TEXT DEFAULT '{}'
            )
        ''')
        
        # Instance tables
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS artifacts (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                type_id INTEGER NOT NULL,
                uri TEXT,
                properties TEXT DEFAULT '{}',
                create_time TEXT DEFAULT CURRENT_TIMESTAMP,
                FOREIGN KEY (type_id) REFERENCES artifact_types(id)
            )
        ''')
        
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS executions (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                type_id INTEGER NOT NULL,
                properties TEXT DEFAULT '{}',
                create_time TEXT DEFAULT CURRENT_TIMESTAMP,
                FOREIGN KEY (type_id) REFERENCES execution_types(id)
            )
        ''')
        
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS contexts (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                type_id INTEGER NOT NULL,
                name TEXT NOT NULL,
                properties TEXT DEFAULT '{}',
                create_time TEXT DEFAULT CURRENT_TIMESTAMP,
                FOREIGN KEY (type_id) REFERENCES context_types(id)
            )
        ''')
        
        # Relationship tables
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS events (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                artifact_id INTEGER NOT NULL,
                execution_id INTEGER NOT NULL,
                type TEXT NOT NULL,
                create_time TEXT DEFAULT CURRENT_TIMESTAMP,
                FOREIGN KEY (artifact_id) REFERENCES artifacts(id),
                FOREIGN KEY (execution_id) REFERENCES executions(id)
            )
        ''')
        
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS attributions (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                artifact_id INTEGER NOT NULL,
                context_id INTEGER NOT NULL,
                FOREIGN KEY (artifact_id) REFERENCES artifacts(id),
                FOREIGN KEY (context_id) REFERENCES contexts(id)
            )
        ''')
        
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS associations (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                execution_id INTEGER NOT NULL,
                context_id INTEGER NOT NULL,
                FOREIGN KEY (execution_id) REFERENCES executions(id),
                FOREIGN KEY (context_id) REFERENCES contexts(id)
            )
        ''')
        
        self.conn.commit()
    
    # --- Type registration methods ---
    
    def put_artifact_type(self, name, properties):
        """Register an artifact type. Returns the type ID."""
        cursor = self.conn.cursor()
        cursor.execute(
            'INSERT INTO artifact_types (name, properties) VALUES (?, ?)',
            (name, json.dumps(properties))
        )
        self.conn.commit()
        return cursor.lastrowid
    
    def put_execution_type(self, name, properties):
        """Register an execution type. Returns the type ID."""
        cursor = self.conn.cursor()
        cursor.execute(
            'INSERT INTO execution_types (name, properties) VALUES (?, ?)',
            (name, json.dumps(properties))
        )
        self.conn.commit()
        return cursor.lastrowid
    
    def put_context_type(self, name, properties):
        """Register a context type. Returns the type ID."""
        cursor = self.conn.cursor()
        cursor.execute(
            'INSERT INTO context_types (name, properties) VALUES (?, ?)',
            (name, json.dumps(properties))
        )
        self.conn.commit()
        return cursor.lastrowid
    
    # --- Instance creation methods ---
    
    def put_artifact(self, type_id, uri, properties):
        """Create an artifact. Returns the artifact ID."""
        cursor = self.conn.cursor()
        cursor.execute(
            'INSERT INTO artifacts (type_id, uri, properties) VALUES (?, ?, ?)',
            (type_id, uri, json.dumps(properties))
        )
        self.conn.commit()
        return cursor.lastrowid
    
    def put_execution(self, type_id, properties):
        """Create an execution. Returns the execution ID."""
        cursor = self.conn.cursor()
        cursor.execute(
            'INSERT INTO executions (type_id, properties) VALUES (?, ?)',
            (type_id, json.dumps(properties))
        )
        self.conn.commit()
        return cursor.lastrowid
    
    def put_context(self, type_id, name, properties):
        """Create a context. Returns the context ID."""
        cursor = self.conn.cursor()
        cursor.execute(
            'INSERT INTO contexts (type_id, name, properties) VALUES (?, ?, ?)',
            (type_id, name, json.dumps(properties))
        )
        self.conn.commit()
        return cursor.lastrowid
    
    # --- Update methods ---
    
    def update_execution(self, execution_id, properties):
        """Update an execution's properties."""
        cursor = self.conn.cursor()
        cursor.execute(
            'UPDATE executions SET properties = ? WHERE id = ?',
            (json.dumps(properties), execution_id)
        )
        self.conn.commit()
    
    # --- Event and relationship methods ---
    
    def put_event(self, artifact_id, execution_id, event_type):
        """Record an event linking an artifact to an execution."""
        cursor = self.conn.cursor()
        cursor.execute(
            'INSERT INTO events (artifact_id, execution_id, type) VALUES (?, ?, ?)',
            (artifact_id, execution_id, event_type)
        )
        self.conn.commit()
    
    def put_attribution(self, artifact_id, context_id):
        """Record an attribution linking an artifact to a context."""
        cursor = self.conn.cursor()
        cursor.execute(
            'INSERT INTO attributions (artifact_id, context_id) VALUES (?, ?)',
            (artifact_id, context_id)
        )
        self.conn.commit()
    
    def put_association(self, execution_id, context_id):
        """Record an association linking an execution to a context."""
        cursor = self.conn.cursor()
        cursor.execute(
            'INSERT INTO associations (execution_id, context_id) VALUES (?, ?)',
            (execution_id, context_id)
        )
        self.conn.commit()
    
    # --- Query methods ---
    
    def get_artifact_types(self):
        """Get all registered artifact types."""
        cursor = self.conn.cursor()
        cursor.execute('SELECT * FROM artifact_types')
        rows = cursor.fetchall()
        return [{'id': r['id'], 'name': r['name'], 'properties': json.loads(r['properties'])} for r in rows]
    
    def get_artifacts_by_type(self, type_name):
        """Get all artifacts of a given type."""
        cursor = self.conn.cursor()
        cursor.execute('''
            SELECT a.*, at.name as type_name 
            FROM artifacts a 
            JOIN artifact_types at ON a.type_id = at.id 
            WHERE at.name = ?
        ''', (type_name,))
        rows = cursor.fetchall()
        return [{'id': r['id'], 'type_id': r['type_id'], 'type': r['type_name'],
                 'uri': r['uri'], 'properties': json.loads(r['properties']),
                 'create_time': r['create_time']} for r in rows]
    
    def get_artifacts_by_id(self, artifact_ids):
        """Get artifacts by their IDs."""
        cursor = self.conn.cursor()
        placeholders = ','.join('?' * len(artifact_ids))
        cursor.execute(f'''
            SELECT a.*, at.name as type_name 
            FROM artifacts a 
            JOIN artifact_types at ON a.type_id = at.id 
            WHERE a.id IN ({placeholders})
        ''', artifact_ids)
        rows = cursor.fetchall()
        return [{'id': r['id'], 'type_id': r['type_id'], 'type': r['type_name'],
                 'uri': r['uri'], 'properties': json.loads(r['properties']),
                 'create_time': r['create_time']} for r in rows]
    
    def get_events_by_artifact_ids(self, artifact_ids):
        """Get all events for given artifact IDs."""
        cursor = self.conn.cursor()
        placeholders = ','.join('?' * len(artifact_ids))
        cursor.execute(f'SELECT * FROM events WHERE artifact_id IN ({placeholders})', artifact_ids)
        rows = cursor.fetchall()
        return [{'artifact_id': r['artifact_id'], 'execution_id': r['execution_id'],
                 'type': r['type'], 'create_time': r['create_time']} for r in rows]
    
    def get_events_by_execution_ids(self, execution_ids):
        """Get all events for given execution IDs."""
        cursor = self.conn.cursor()
        placeholders = ','.join('?' * len(execution_ids))
        cursor.execute(f'SELECT * FROM events WHERE execution_id IN ({placeholders})', execution_ids)
        rows = cursor.fetchall()
        return [{'artifact_id': r['artifact_id'], 'execution_id': r['execution_id'],
                 'type': r['type'], 'create_time': r['create_time']} for r in rows]
    
    def get_artifacts_by_context(self, context_id):
        """Get all artifacts linked to a context via attributions."""
        cursor = self.conn.cursor()
        cursor.execute('''
            SELECT a.*, at.name as type_name 
            FROM artifacts a
            JOIN artifact_types at ON a.type_id = at.id
            JOIN attributions attr ON a.id = attr.artifact_id
            WHERE attr.context_id = ?
        ''', (context_id,))
        rows = cursor.fetchall()
        return [{'id': r['id'], 'type_id': r['type_id'], 'type': r['type_name'],
                 'uri': r['uri'], 'properties': json.loads(r['properties']),
                 'create_time': r['create_time']} for r in rows]
    
    def get_executions_by_context(self, context_id):
        """Get all executions linked to a context via associations."""
        cursor = self.conn.cursor()
        cursor.execute('''
            SELECT e.*, et.name as type_name
            FROM executions e
            JOIN execution_types et ON e.type_id = et.id
            JOIN associations assoc ON e.id = assoc.execution_id
            WHERE assoc.context_id = ?
        ''', (context_id,))
        rows = cursor.fetchall()
        return [{'id': r['id'], 'type_id': r['type_id'], 'type': r['type_name'],
                 'properties': json.loads(r['properties']),
                 'create_time': r['create_time']} for r in rows]

print('MetadataStore class defined successfully.')

## Load and Preview the Dataset

We use the **Student Performance in Exams** dataset with features like gender, parental education, and test scores. The data is already split into train, eval, and serving sets.

In [ ]:
# Verify data files exist
for split in ['train', 'eval', 'serving']:
    path = f'./data/{split}/data.csv'
    df = pd.read_csv(path)
    print(f'{split}: {len(df)} rows, {len(df.columns)} columns')

print('\nColumn names:', list(df.columns))

In [ ]:
# Preview the training data
train_df = pd.read_csv('./data/train/data.csv')
train_df.head(10)

In [ ]:
# Quick summary statistics
train_df.describe(include='all')

## Step 1: Define the Metadata Store

We instantiate our custom SQLite-based metadata store. Using `:memory:` creates an in-memory database (similar to MLMD's fake database). In production, you would use a file path for persistence.

In [ ]:
# Create an in-memory metadata store (similar to MLMD's fake database)
store = MetadataStore(db_path=':memory:')

print('Metadata store created successfully (in-memory SQLite backend).')

## Step 2: Register Artifact Types

We define artifact types for the objects in our pipeline:
- **DataSet**: The input CSV data with properties for name, split, and version
- **Schema**: The output schema file with properties for name and version
- **Statistics**: Summary statistics of the dataset

In [ ]:
# Register ArtifactType for Statistics
statistics_type_id = store.put_artifact_type(
    name='Statistics',
    properties={'name': 'STRING', 'split': 'STRING', 'version': 'INT'}
)
print(f'Statistics artifact type registered. ID: {statistics_type_id}')

# Register ArtifactType for DataSet
data_type_id = store.put_artifact_type(
    name='DataSet',
    properties={'name': 'STRING', 'split': 'STRING', 'version': 'INT'}
)
print(f'DataSet artifact type registered. ID: {data_type_id}')

# Register ArtifactType for Schema
schema_type_id = store.put_artifact_type(
    name='Schema',
    properties={'name': 'STRING', 'version': 'INT'}
)
print(f'Schema artifact type registered. ID: {schema_type_id}')

# Verify all registered types
print('\nAll registered artifact types:')
for at in store.get_artifact_types():
    print(f"  ID: {at['id']}, Name: {at['name']}, Properties: {at['properties']}")

## Step 3: Register Execution Type

We create an execution type for our **Data Validation** step. This represents the pipeline component that reads the dataset and generates a schema.

In [ ]:
# Register ExecutionType for Data Validation
dv_execution_type_id = store.put_execution_type(
    name='Data Validation',
    properties={'state': 'STRING'}
)

print(f'Data Validation execution type registered.')
print(f'Execution type ID: {dv_execution_type_id}')

## Step 4: Generate Input Artifact

We create a specific artifact instance for our training dataset. This records the URI (file path), dataset name, split type, and version.

In [ ]:
# Create input artifact for the training dataset
data_artifact_id = store.put_artifact(
    type_id=data_type_id,
    uri='./data/train/data.csv',
    properties={
        'name': 'Student Performance dataset',
        'split': 'train',
        'version': 1
    }
)

print(f'Data artifact created.')
print(f'Artifact ID: {data_artifact_id}')
print(f'Type ID: {data_type_id}')
print(f'URI: ./data/train/data.csv')
print(f'Properties: name=Student Performance dataset, split=train, version=1')

## Step 5: Generate Execution Unit

We create an execution instance for the Data Validation run. The state is initially set to `RUNNING`.

In [ ]:
# Create execution for data validation
dv_execution_id = store.put_execution(
    type_id=dv_execution_type_id,
    properties={'state': 'RUNNING'}
)

print(f'Data Validation execution created.')
print(f'Execution ID: {dv_execution_id}')
print(f'State: RUNNING')

## Step 6: Register Input Event

An **Event** links an artifact to an execution. Here we declare that the dataset artifact is a `DECLARED_INPUT` to the data validation execution.

In [ ]:
# Record input event: dataset -> data validation
store.put_event(
    artifact_id=data_artifact_id,
    execution_id=dv_execution_id,
    event_type='DECLARED_INPUT'
)

print(f'Input event registered:')
print(f'  Artifact ID: {data_artifact_id} (DataSet)')
print(f'  Execution ID: {dv_execution_id} (Data Validation)')
print(f'  Type: DECLARED_INPUT')

## Step 7: Run the Schema Generation Component

Instead of using TFDV (as in the original lab), we use **pandas** to infer the schema from the training data. Our custom function analyzes each column to determine its type, presence requirements, and domain values.

In [ ]:
def generate_schema_from_csv(csv_path, schema_output_path):
    """Generate a schema.pbtxt-style file from a CSV using pandas.
    
    Analyzes each column to determine:
    - Data type (INT, FLOAT, BYTES for strings)
    - Presence requirements (min_fraction based on null count)
    - Domain values for categorical string columns
    
    Args:
        csv_path: Path to the input CSV file
        schema_output_path: Path to write the schema file
    
    Returns:
        The schema text content
    """
    df = pd.read_csv(csv_path)
    
    schema_lines = []
    string_domains = {}
    
    for col in df.columns:
        schema_lines.append('feature {')
        schema_lines.append(f'  name: "{col}"')
        
        # Determine type based on pandas dtype
        if df[col].dtype == 'int64':
            schema_lines.append('  type: INT')
        elif df[col].dtype == 'float64':
            schema_lines.append('  type: FLOAT')
        else:
            schema_lines.append('  type: BYTES')
            # Collect domain values for string/categorical columns
            domain_name = col.replace(' ', '_').replace('/', '_')
            string_domains[domain_name] = sorted(df[col].dropna().unique().tolist())
            schema_lines.append(f'  domain: "{domain_name}"')
        
        # Calculate presence based on null count
        min_fraction = 1.0 - (df[col].isnull().sum() / len(df))
        schema_lines.append('  presence {')
        schema_lines.append(f'    min_fraction: {min_fraction:.1f}')
        schema_lines.append('    min_count: 1')
        schema_lines.append('  }')
        
        schema_lines.append('  shape {')
        schema_lines.append('    dim {')
        schema_lines.append('      size: 1')
        schema_lines.append('    }')
        schema_lines.append('  }')
        schema_lines.append('}')
    
    # Add string domain definitions
    for domain_name, values in string_domains.items():
        schema_lines.append('string_domain {')
        schema_lines.append(f'  name: "{domain_name}"')
        for val in values:
            schema_lines.append(f'  value: "{val}"')
        schema_lines.append('}')
    
    schema_text = '\n'.join(schema_lines)
    
    with open(schema_output_path, 'w') as f:
        f.write(schema_text)
    
    return schema_text

# Run the schema generation
train_data = './data/train/data.csv'
schema_file = './schema.pbtxt'

schema_text = generate_schema_from_csv(train_data, schema_file)

print(f"Schema generated successfully at: {schema_file}")
print(f"\n{'='*50}")
print("Generated Schema (first 40 lines):")
print('='*50)
print('\n'.join(schema_text.split('\n')[:40]))
print('...')

## Step 8: Generate Output Artifact

Now that the schema has been generated, we create an artifact to represent it in the metadata store.

In [ ]:
# Create output artifact for the schema
schema_artifact_id = store.put_artifact(
    type_id=schema_type_id,
    uri='./schema.pbtxt',
    properties={
        'name': 'Student Performance Schema',
        'version': 1
    }
)

print(f'Schema artifact created.')
print(f'Artifact ID: {schema_artifact_id}')
print(f'Type ID: {schema_type_id}')
print(f'URI: ./schema.pbtxt')
print(f'Properties: name=Student Performance Schema, version=1')

## Step 9: Register Output Event

We record that the schema artifact is a `DECLARED_OUTPUT` of the data validation execution.

In [ ]:
# Record output event: data validation -> schema
store.put_event(
    artifact_id=schema_artifact_id,
    execution_id=dv_execution_id,
    event_type='DECLARED_OUTPUT'
)

print(f'Output event registered:')
print(f'  Artifact ID: {schema_artifact_id} (Schema)')
print(f'  Execution ID: {dv_execution_id} (Data Validation)')
print(f'  Type: DECLARED_OUTPUT')

## Step 10: Update the Execution Unit

Since the data validation step has finished successfully, we update the state from `RUNNING` to `COMPLETED`.

In [ ]:
# Update execution state to COMPLETED
store.update_execution(
    execution_id=dv_execution_id,
    properties={'state': 'COMPLETED'}
)

print(f'Execution ID {dv_execution_id} updated.')
print(f'State: COMPLETED')

## Step 11: Set Up Context Type and Generate Context

A **Context** groups related artifacts and executions. We create an `Experiment` context to represent this entire lab run.

In [ ]:
# Register context type
expt_context_type_id = store.put_context_type(
    name='Experiment',
    properties={'note': 'STRING'}
)
print(f'Experiment context type registered. ID: {expt_context_type_id}')

# Create context instance
expt_context_id = store.put_context(
    type_id=expt_context_type_id,
    name='Student Performance Experiment',
    properties={'note': 'MLMD lab using Student Performance dataset with pandas-based schema validation'}
)

print(f'Experiment context created. ID: {expt_context_id}')
print(f'Name: Student Performance Experiment')

## Step 12: Generate Attribution and Association

We link the schema artifact and data validation execution to the experiment context:
- **Attribution**: Links the schema artifact to the experiment context
- **Association**: Links the data validation execution to the experiment context

In [ ]:
# Create attribution: schema artifact <-> experiment context
store.put_attribution(
    artifact_id=schema_artifact_id,
    context_id=expt_context_id
)
print(f'Attribution created: Artifact {schema_artifact_id} <-> Context {expt_context_id}')

# Create association: execution <-> experiment context
store.put_association(
    execution_id=dv_execution_id,
    context_id=expt_context_id
)
print(f'Association created: Execution {dv_execution_id} <-> Context {expt_context_id}')

---

## Retrieving Information from the Metadata Store

Now that we've recorded everything, let's query the metadata store to trace lineage. We'll investigate: **which dataset was used to generate the schema?**

This demonstrates the core value of MLMD — even without seeing the original code, we can trace any artifact back to its source.

In [ ]:
# Step A: Get all registered artifact types
print('=== All Registered Artifact Types ===')
for at in store.get_artifact_types():
    print(f"  ID: {at['id']}, Name: {at['name']}, Properties: {at['properties']}")

In [ ]:
# Step B: Get the schema artifact
schema_artifacts = store.get_artifacts_by_type('Schema')
schema_to_inv = schema_artifacts[0]

print('=== Schema Artifact to Investigate ===')
print(f"  ID: {schema_to_inv['id']}")
print(f"  Type: {schema_to_inv['type']}")
print(f"  URI: {schema_to_inv['uri']}")
print(f"  Properties: {schema_to_inv['properties']}")
print(f"  Created: {schema_to_inv['create_time']}")

In [ ]:
# Step C: Find events linked to this schema
schema_events = store.get_events_by_artifact_ids([schema_to_inv['id']])

print('=== Events Linked to Schema Artifact ===')
for event in schema_events:
    print(f"  Artifact ID: {event['artifact_id']}, Execution ID: {event['execution_id']}, Type: {event['type']}")

In [ ]:
# Step D: The schema is an output — find ALL events for that execution
execution_events = store.get_events_by_execution_ids([schema_events[0]['execution_id']])

print('=== All Events for the Execution That Produced the Schema ===')
for event in execution_events:
    print(f"  Artifact ID: {event['artifact_id']}, Type: {event['type']}")

In [ ]:
# Step E: Look up the INPUT artifact — this is the dataset used to generate the schema!
input_events = [e for e in execution_events if e['type'] == 'DECLARED_INPUT']
input_artifact = store.get_artifacts_by_id([input_events[0]['artifact_id']])[0]

print('=' * 55)
print('LINEAGE RESULT: Dataset used to generate the schema')
print('=' * 55)
print(f"  Artifact ID: {input_artifact['id']}")
print(f"  Type: {input_artifact['type']}")
print(f"  URI: {input_artifact['uri']}")
print(f"  Name: {input_artifact['properties']['name']}")
print(f"  Split: {input_artifact['properties']['split']}")
print(f"  Version: {input_artifact['properties']['version']}")

We successfully traced the lineage from the **Schema** back to the **Student Performance training dataset** that was used to generate it!

## Query Artifacts and Executions by Context

In [ ]:
# Get all artifacts linked to our experiment context
experiment_artifacts = store.get_artifacts_by_context(expt_context_id)
print('=== Artifacts in Experiment Context ===')
for artifact in experiment_artifacts:
    print(f"  Artifact ID: {artifact['id']}, Type: {artifact['type']}, URI: {artifact['uri']}")

print()

# Get all executions linked to our experiment context
experiment_executions = store.get_executions_by_context(expt_context_id)
print('=== Executions in Experiment Context ===')
for execution in experiment_executions:
    print(f"  Execution ID: {execution['id']}, Type: {execution['type']}, State: {execution['properties']['state']}")

## Wrap Up

In this lab, we demonstrated how to use **ML Metadata** concepts to track the lineage of an ML pipeline using the **Student Performance in Exams** dataset. Key takeaways:

1. **MLMD concepts can be implemented independently** — we built a custom SQLite-based metadata store that mirrors the MLMD API
2. **Artifact types, execution types, and context types** define the structure of your metadata
3. **Events** link artifacts to executions as inputs and outputs
4. **Contexts with attributions and associations** group related pipeline components
5. **Lineage queries** let you trace any artifact back to its origin — we traced from schema to the original dataset

This pattern is essential in production ML systems where you need to track which data, code, and parameters produced a given model.